In [10]:
import pandas as pd
import numpy as np
import json
import warnings

# Suppress warnings for cleaner output + warning annoy me
warnings.filterwarnings('ignore')

# Load the Excel file
df_raw = pd.read_excel('/home/tim/Documents/Studies/CDU/PRT564/Assignments/Assignment2/Data1.xlsx', sheet_name='Sheet1', header=None)


# PART 1: Extract and preserve metadata (rows 2-10)


# Metadata rows in Excel (rows 2-10, which are indices 1-9 in pandas)
metadata_rows = df_raw.iloc[1:10, :].copy()

# The first column contains the metadata labels
metadata_labels = metadata_rows.iloc[:, 0].tolist()
metadata_values = metadata_rows.iloc[:, 1:].values

# Create a metadata dictionary
metadata_dict = {}
for i, label in enumerate(metadata_labels):
    if pd.notna(label):
        values = metadata_values[i, :]
        clean_values = [v for v in values if pd.notna(v)]
        metadata_dict[str(label).strip()] = clean_values

# Create metadata DataFrame
metadata_df = pd.DataFrame()
for label, values in metadata_dict.items():
    metadata_df[label] = pd.Series(values)

print("=" * 80)
print("METADATA EXTRACTED")
print("=" * 80)
print(f" Metadata labels found: {list(metadata_dict.keys())}")


# PART 2: Extract column headers


headers = df_raw.iloc[0, 1:].fillna('').astype(str)

# Create unique column names
column_names = ['Date']
header_counts = {}

for header in headers:
    if header and header != 'nan':
        if header in header_counts:
            header_counts[header] += 1
            unique_header = f"{header}_{header_counts[header]}"
        else:
            header_counts[header] = 0
            unique_header = header       
        column_names.append(unique_header)
    else:
        column_names.append(f'Column_{len(column_names)}')

print(f" Created {len(column_names)} column names")


# PART 3: Extract data


# Find where data actually starts
data_start_idx = 10
for idx in range(10, len(df_raw)):
    cell_value = df_raw.iloc[idx, 0]
    if isinstance(cell_value, str) and '1998' in cell_value:
        data_start_idx = idx
        break
    elif hasattr(cell_value, 'year') and cell_value.year >= 1998:
        data_start_idx = idx
        break

print(f" Data starts at row index {data_start_idx}")

# Extract data
df_data = df_raw.iloc[data_start_idx:, :].copy()
df_data.columns = column_names

# Clean date column
df_data['Date'] = pd.to_datetime(df_data['Date'], errors='coerce')

# Remove rows with invalid dates
df_clean = df_data.dropna(subset=['Date']).reset_index(drop=True)

# Convert all other columns to numeric
for col in df_clean.columns[1:]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Sort by date
df_clean = df_clean.sort_values('Date').reset_index(drop=True)


# PART 4: Save files (with fallback options)


# Save metadata as JSON (always works)
with open('metadata.json', 'w') as f:
    json.dump(metadata_dict, f, indent=2, default=str)

# Save metadata as CSV (always works)
#metadata_df.to_csv('metadata.csv', index=False)

# Try to save main data as Parquet, fallback to CSV if needed
parquet_success = False
try:
    # Try different parquet engines
    for engine in ['pyarrow', 'fastparquet']:
        try:
            df_clean.to_parquet('cleaned_data.parquet', index=False, engine=engine)
            parquet_success = True
            print(f" Saved as Parquet using {engine} engine")
            break
        except Exception as e:
            print(f"  {engine} failed: {str(e)[:50]}...")
            continue
except Exception as e:
    print(f"  Parquet save failed: {e}")

# Always save as CSV as backup
#df_clean.to_csv('cleaned_data.csv', index=False)
#print(" Saved as CSV (backup): cleaned_data.csv")

# Also save as Feather if available (alternative to Parquet)
#try:
 #   df_clean.to_feather('cleaned_data.feather')
  #  print(" Saved as Feather: cleaned_data.feather")
#except:
  #  pass

print("\n" + "=" * 80)
print("FILES SAVED SUCCESSFULLY")
print("=" * 80)
print(f" Main data shape: {df_clean.shape}")
print(f" Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")
print(f" Metadata (JSON): metadata.json")
#print(f" Metadata (CSV): metadata.csv")

if parquet_success:
    print(" Main data (Parquet): cleaned_data.parquet")
#print(" Main data (CSV): cleaned_data.csv")


# PART 5: Summary statistics


print("\n DATA SUMMARY:")

# Column counts by type
index_cols = [col for col in df_clean.columns if 'Index Numbers' in col and 'Change in Points' not in col]
pct_change_cols = [col for col in df_clean.columns if 'Percentage Change' in col]
points_contrib_cols = [col for col in df_clean.columns if 'Points Contribution' in col and 'Change in Points' not in col]
change_points_cols = [col for col in df_clean.columns if 'Change in Points Contribution' in col]

print(f"   Index Numbers columns: {len(index_cols)}")
print(f"   Percentage Change columns: {len(pct_change_cols)}")
print(f"   Points Contribution columns: {len(points_contrib_cols)}")
print(f"   Change in Points Contribution columns: {len(change_points_cols)}")
print(f"   Total data columns: {len(df_clean.columns)-1}")

print("\n Sample of Change in Points Contribution columns:")
for i, col in enumerate(change_points_cols[:12]):
    display_col = col[:80] + '...' if len(col) > 80 else col
    print(f"   {i+1:2d}. {display_col}")


# PART 6: Loading instructions - Probabky will delete


print("\n" + "=" * 80)
print("HOW TO LOAD THE DATA BACK:")
print("=" * 80)
print("""
# Load main data (CSV - always works):
df = pd.read_csv('cleaned_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# OR if Parquet worked:
# df = pd.read_parquet('cleaned_data.parquet')

# Load metadata:
import json
with open('metadata.json', 'r') as f:
    metadata = json.load(f)

# Or load metadata as DataFrame:
#metadata_df = pd.read_csv('metadata.csv')
""")


# PART 7: Quick verification


print("\n FIRST 3 ROWS (first 5 columns):")
print(df_clean.iloc[:3, :5].to_string())

print("\n LAST 3 ROWS (last 5 columns):")
print(df_clean.iloc[-3:, -5:].to_string())

METADATA EXTRACTED
 Metadata labels found: ['Unit', 'Series Type', 'Data Type', 'Frequency', 'Collection Month', 'Series Start', 'Series End', 'No. Obs', 'Series ID']
 Created 251 column names
 Data starts at row index 10
 Saved as Parquet using pyarrow engine

FILES SAVED SUCCESSFULLY
 Main data shape: (111, 251)
 Date range: 1998-06-01 00:00:00 to 2025-12-01 00:00:00
 Metadata (JSON): metadata.json
 Main data (Parquet): cleaned_data.parquet

 DATA SUMMARY:
   Index Numbers columns: 60
   Percentage Change columns: 120
   Points Contribution columns: 60
   Change in Points Contribution columns: 10
   Total data columns: 250

 Sample of Change in Points Contribution columns:
    1. Change in Points Contribution ;  Pensioner and beneficiary households ;  All gro...
    2. Change in Points Contribution ;  Pensioner and beneficiary households ;  Food an...
    3. Change in Points Contribution ;  Pensioner and beneficiary households ;  Alcohol...
    4. Change in Points Contribution ;  Pen

In [11]:
import pandas as pd

df = pd.read_parquet('cleaned_data.parquet')

# Set Date as index first — keeps it out of the MultiIndex parsing
df = df.set_index('Date')

# Parse each column name into 3 parts
tuples = []
for col in df.columns:
    parts = [p.strip() for p in col.split(';') if p.strip()]
    if len(parts) == 3:
        tuples.append((parts[0], parts[1], parts[2]))
    else:
        # Fallback — shouldn't happen but good to know if it does
        print(f'Unexpected format: {repr(col)}')
        tuples.append((col, '', ''))

df.columns = pd.MultiIndex.from_tuples(tuples, names=['Measure', 'Household', 'Commodity'])

print(df.columns.get_level_values('Measure').unique().tolist())
print(df.columns.get_level_values('Household').unique().tolist())
print(df.columns.get_level_values('Commodity').unique().tolist())

# Save
df.to_parquet('cleaned_data_multiindex.parquet')

['Index Numbers', 'Percentage Change from Corresponding Quarter of Previous Year', 'Percentage Change from Previous Period', 'Points Contribution to All Groups', 'Change in Points Contribution']
['Pensioner and beneficiary households', 'Employee households', 'Age pensioner households', 'Other government transfer recipient households', 'Self-funded retiree households']
['All groups', 'Food and non-alcoholic beverages', 'Alcohol and tobacco', 'Clothing and footwear', 'Housing', 'Furnishings, household equipment and services', 'Health', 'Transport', 'Communication', 'Recreation and culture', 'Education', 'Insurance and financial services']


In [9]:
df = pd.read_parquet('cleaned_data_multiindex.parquet')
print(df.index[:3])        # Date is here
print(df.columns[:3])      # MultiIndex tuples are here
print(df.shape)            # (111, 250)


DatetimeIndex(['1998-06-01', '1998-09-01', '1998-12-01'], dtype='datetime64[us]', name='Date', freq=None)
MultiIndex([('Index Numbers', 'Pensioner and beneficiary households', ...),
            ('Index Numbers', 'Pensioner and beneficiary households', ...),
            ('Index Numbers', 'Pensioner and beneficiary households', ...)],
           names=['Measure', 'Household', 'Commodity'])
(111, 250)


In [12]:
df = pd.read_parquet('cleaned_data_multiindex.parquet')
print(df.index[:3])
print(df.columns[:3])
print(df.shape)

pd.set_option('display.max_rows', 100)
print(df.head(100))

DatetimeIndex(['1998-06-01', '1998-09-01', '1998-12-01'], dtype='datetime64[us]', name='Date', freq=None)
MultiIndex([('Index Numbers', 'Pensioner and beneficiary households', ...),
            ('Index Numbers', 'Pensioner and beneficiary households', ...),
            ('Index Numbers', 'Pensioner and beneficiary households', ...)],
           names=['Measure', 'Household', 'Commodity'])
(111, 250)
Measure                           Index Numbers  \
Household  Pensioner and beneficiary households   
Commodity                            All groups   
Date                                              
1998-06-01                                  NaN   
1998-09-01                                  NaN   
1998-12-01                                  NaN   
1999-03-01                                  NaN   
1999-06-01                                  NaN   
1999-09-01                                  NaN   
1999-12-01                                  NaN   
2000-03-01                           